In [ ]:
#Qué hace: identifica nuevos cultivos positivos durante el seguimiento diario.
#Clave: evento microbiológico observado ≠ nueva infección clínica.

In [1]:
# 0. setup 

from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = "mimic-pruebas"
HOSP = "physionet-data.mimiciv_3_1_hosp"
ICU  = "physionet-data.mimiciv_3_1_icu"
DERIVED = "physionet-data.mimiciv_3_1_derived"

client = bigquery.Client(project=PROJECT_ID)

WINDOWS_PATH = "05_ventanas_24h.parquet"
COHORT_PATH  = "04_cohorte_base_T0.parquet"   # recomendado
OUT_PATH     = "14_new_positive_cultures_daily.parquet"

# Elegimos tiempo principal:
MICRO_TIME_COL = "charttime"   # alternativa: "storetime"

In [2]:
# 1 Inputs: ventanas + cohorte + baseline (sitio/germen inicial)

df_win = pd.read_parquet(WINDOWS_PATH)[
    ["subject_id","hadm_id","icu_stay_id","day_idx","window_start","window_end"]
].copy()

df_base = pd.read_parquet(COHORT_PATH).copy()

print("Baseline columns:", df_base.columns.tolist())

# TODO: ajusta estos nombres a los reales de tu df_base
COL_INITIAL_SITE = "site"
COL_INITIAL_ORG  = "organism"   # o "organism_name" / "microorganism"

df_init = df_base[["subject_id","hadm_id","icu_stay_id", COL_INITIAL_SITE, COL_INITIAL_ORG]].drop_duplicates()

Baseline columns: ['subject_id', 'hadm_id', 'icu_stay_id', 'infection_time', 't0_antibiotic', 'organism', 'pathogen_group', 'site', 'age', 'gender', 'icu_in', 'icu_out']


In [3]:
# 2 Extraer microeventos “positivos”

df_keys = df_win[["subject_id","hadm_id"]].drop_duplicates()

# Para meter df_keys en SQL sin líos, hacemos un IN por hadm_id (rápido).
hadm_ids = df_keys["hadm_id"].dropna().astype(int).unique().tolist()

# Si son demasiados hadm_id, te digo cómo hacerlo con tabla temporal; de momento probamos.
hadm_ids_sql = ",".join(map(str, hadm_ids[:20000]))  # safety cap; si >20k, lo cambiamos

sql_micro = f"""
SELECT
  microevent_id,
  subject_id,
  hadm_id,
  {MICRO_TIME_COL} AS event_time,
  spec_type_desc,
  org_name,
  interpretation
FROM `{HOSP}.microbiologyevents`
WHERE hadm_id IN ({hadm_ids_sql})
  AND org_name IS NOT NULL
"""
df_micro = client.query(sql_micro).to_dataframe()

df_micro["event_time"] = pd.to_datetime(df_micro["event_time"])
df_micro["spec_type_desc"] = df_micro["spec_type_desc"].astype(str)
df_micro["org_name"] = df_micro["org_name"].astype(str)

E0000 00:00:1769606185.328751  952915 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [4]:
# 3 Filtrar a patógenos del estudio (muy importante)

# Patrones aproximados
PATTERNS = [
    # Enterobacterales
    "ESCHERICHIA COLI",
    "KLEBSIELLA",
    "ENTEROBACTER",
    "SERRATIA",
    "CITROBACTER",
    "PROTEUS",

    # Non-fermenters
    "PSEUDOMONAS AERUGINOSA",
    "ACINETOBACTER",
    "STENOTROPHOMONAS MALTOPHILIA",

    # Gram positives
    "ENTEROCOCCUS FAECIUM",
    "STAPHYLOCOCCUS AUREUS",
]


org_upper = df_micro["org_name"].str.upper()
mask_path = np.zeros(len(df_micro), dtype=bool)
for p in PATTERNS:
    mask_path |= org_upper.str.contains(p, na=False)

df_micro = df_micro[mask_path].copy()

In [5]:
# 4 Asignar cada microevento a un day_idx por ventanas

df_m = df_win.merge(
    df_micro,
    on=["subject_id","hadm_id"],
    how="left"
)

mask_in_window = (
    (df_m["event_time"] >= df_m["window_start"]) &
    (df_m["event_time"] <  df_m["window_end"])
)

df_m = df_m[mask_in_window].copy()

In [6]:
# 5 Definición de “nuevo positivo relevante” 
# Un evento del día cuenta como “nuevo” si cumple al menos uno:
# sitio distinto al inicial
# germen distinto en el mismo sitio
# mismo germen en otro sitio
# hemocultivo positivo en evolución
# Necesitamos definir “blood culture” por spec_type_desc. En MIMIC suele aparecer como contiene “BLOOD”.

df_m = df_m.merge(df_init, on=["subject_id","hadm_id","icu_stay_id"], how="left")

# Normalizaciones
df_m["site_init"] = df_m[COL_INITIAL_SITE].astype(str).str.upper().str.strip()
df_m["org_init"]  = df_m[COL_INITIAL_ORG].astype(str).str.upper().str.strip()

df_m["site_evt"]  = df_m["spec_type_desc"].astype(str).str.upper().str.strip()
df_m["org_evt"]   = df_m["org_name"].astype(str).str.upper().str.strip()

is_blood = df_m["site_evt"].str.contains("BLOOD", na=False)

new_site = (df_m["site_evt"] != df_m["site_init"])
new_org_same_site = (df_m["site_evt"] == df_m["site_init"]) & (df_m["org_evt"] != df_m["org_init"])
same_org_new_site = (df_m["org_evt"] == df_m["org_init"]) & (df_m["site_evt"] != df_m["site_init"])

df_m["blood_culture_flag_evt"] = is_blood.astype(int)
df_m["new_site_flag_evt"] = new_site.astype(int)
df_m["new_org_same_site_flag_evt"] = new_org_same_site.astype(int)
df_m["same_org_new_site_flag_evt"] = same_org_new_site.astype(int)

df_m["new_pos_relevant_evt"] = (
    is_blood | new_site | new_org_same_site | same_org_new_site
).astype(int)

# Agregación diaria por estancia:

df_day = (
    df_m.groupby(["subject_id","hadm_id","icu_stay_id","day_idx"], as_index=False)
        .agg(
            new_pos_culture_flag=("new_pos_relevant_evt","max"),
            blood_culture_flag=("blood_culture_flag_evt","max"),
            new_site_flag=("new_site_flag_evt","max"),
            new_org_same_site_flag=("new_org_same_site_flag_evt","max"),
            same_org_new_site_flag=("same_org_new_site_flag_evt","max"),
            n_microevents=("microevent_id","nunique")
        )
)

# Completar días sin microeventos:
df_out = df_win.merge(df_day, on=["subject_id","hadm_id","icu_stay_id","day_idx"], how="left")

for c in ["new_pos_culture_flag","blood_culture_flag","new_site_flag","new_org_same_site_flag","same_org_new_site_flag","n_microevents"]:
    df_out[c] = df_out[c].fillna(0).astype(int)

In [7]:
# 6. guardar

keep_cols = [
    "subject_id","hadm_id","icu_stay_id","day_idx",
    "new_pos_culture_flag",
    "blood_culture_flag","new_site_flag","new_org_same_site_flag","same_org_new_site_flag",
    "n_microevents"
]
df_out[keep_cols].to_parquet(OUT_PATH, index=False)
df_out[keep_cols].head()

,subject_id,hadm_id,icu_stay_id,day_idx,new_pos_culture_flag,blood_culture_flag,new_site_flag,new_org_same_site_flag,same_org_new_site_flag,n_microevents
0,11239107,25883588,32367987,0,0,0,0,0,0,0
1,11239107,25883588,32367987,1,0,0,0,0,0,0
2,11239107,25883588,32367987,2,0,0,0,0,0,0
3,11239107,25883588,32367987,3,0,0,0,0,0,0
4,11239107,25883588,32367987,4,0,0,0,0,0,0


In [8]:
print("Pct days with any micro event:", (df_out["n_microevents"] > 0).mean())
print("Pct days with new_pos_culture_flag:", df_out["new_pos_culture_flag"].mean())

print(df_out["n_microevents"].describe())
print(df_out["new_pos_culture_flag"].value_counts(dropna=False).head())

print("df_win:", df_win.shape)
print("df_keys:", df_keys.shape)
print("df_micro (raw):", df_micro.shape)

# tras filtro de patógenos
print("df_micro (after pathogen filter):", df_micro.shape)

# tras merge ventanas x micro
print("df_m (after merge):", df_m.shape)

# tras filtrar por ventana temporal
print("df_m (in-window):", df_m.shape)

Pct days with any micro event: 0.05660218819929065
Pct days with new_pos_culture_flag: 0.056381068626670555
count    1.130610e+06
mean     4.984654e-01
std      2.685103e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.100000e+01
Name: n_microevents, dtype: float64
new_pos_culture_flag
0    1066865
1      63745
Name: count, dtype: int64
df_win: (1130610, 6)
df_keys: (18190, 2)
df_micro (raw): (133692, 7)
df_micro (after pathogen filter): (133692, 7)
df_m (after merge): (5043119, 22)
df_m (in-window): (5043119, 22)


In [9]:
df_micro["org_name"].value_counts().head(30)

org_name
ESCHERICHIA COLI                             43876
KLEBSIELLA PNEUMONIAE                        23797
PSEUDOMONAS AERUGINOSA                       23650
PROTEUS MIRABILIS                             6505
SERRATIA MARCESCENS                           5744
ENTEROBACTER CLOACAE COMPLEX                  5542
ENTEROCOCCUS FAECIUM                          4943
KLEBSIELLA OXYTOCA                            3969
ENTEROBACTER AEROGENES                        3092
ACINETOBACTER BAUMANNII COMPLEX               2889
ENTEROBACTER CLOACAE                          1926
CITROBACTER FREUNDII COMPLEX                  1662
STENOTROPHOMONAS MALTOPHILIA                  1509
NON-FERMENTER, NOT PSEUDOMONAS AERUGINOSA     1073
CITROBACTER KOSERI                            1036
ACINETOBACTER BAUMANNII                        634
PROTEUS VULGARIS                               279
ENTEROBACTER ASBURIAE                          202
KLEBSIELLA (RAOULTELLA) PLANTICOLA             150
ACINETOBACTER SP.     

In [10]:
# Note: pathogen filtering does not reduce the dataset further,
# as all post-T0 positive cultures in the cohort correspond to
# predefined clinically relevant pathogens.